<a href="https://colab.research.google.com/github/ghanist25/CEI_Assignment/blob/main/Week7_GhanistAgrawal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering System (RAG)

**Topic:** Retrieval-Augmented Generation on custom notes

Basic idea - instead of asking a model a question and hoping it "remembers"
the right thing, give it the actual document first and let it answer from
that. So the pipeline here is:

1. Take a document (using my own DBMS notes as the dataset for this)
2. Split it into small chunks
3. Convert every chunk into a vector (TF-IDF, kept this part simple)
4. When a question comes in, find the chunks most similar to the question
5. Feed those chunks + the question into a language model and get the answer

That's the Retrieval + Augmented + Generation part - retrieve the right
context, augment the prompt with it, generate the answer using that context
instead of the model just guessing from what it already knows.

Didn't use LangChain or any RAG framework here on purpose - wanted to build
each piece myself so I actually understand what's happening at every step
instead of one function doing everything behind the scenes.

## 1. Imports

In [2]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.7 MB/s eta 0:00:00


In [3]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# only needed if the input doc is a pdf instead of plain txt
from pypdf import PdfReader

## 2. Loading the document

Using my DBMS notes (`dbms_notes.txt`) as the custom dataset here since the
assignment allows using our own notes/resume instead of a dataset from
Hugging Face.

Wrote the loader to handle both txt and pdf so I don't need to change this
part if I swap the file later.

In [5]:
from google.colab import files
uploaded = files.upload()

Saving dbms_notes.txt to dbms_notes.txt


In [6]:
def load_document(path):
    if path.endswith(".pdf"):
        reader = PdfReader(path)
        full_text = ""
        for page in reader.pages:
            full_text += page.extract_text() + "\n"
        return full_text
    else:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

DOC_PATH = "dbms_notes.txt"
raw_text = load_document(DOC_PATH)

print("total characters in document:", len(raw_text))
print(raw_text[:400])

total characters in document: 4473
Database Management System Notes

Chapter 1: Introduction to DBMS
A database management system (DBMS) is software that is used to create and manage databases. It provides an interface between the user and the database, allowing users to store, retrieve, update and delete data efficiently. Unlike a traditional file system, a DBMS reduces data redundancy and improves data integrity through centraliz


## 3. Chunking the text

A model can't take the whole document in one go (and even if it technically
could, it's wasteful), so I'm splitting it into smaller chunks. Doing simple
fixed length chunking with some overlap between consecutive chunks so a
sentence doesn't get cut off right at a chunk boundary and lose its meaning.

- `chunk_size` - how many characters per chunk
- `overlap` - how many characters repeat between one chunk and the next

Tried a couple of sizes, 400 with 50 overlap gave decent looking chunks for
this document without being too small or too big.

In [7]:
def chunk_text(text, chunk_size=400, overlap=50):
    text = text.strip()
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if len(chunk) > 20:   # skip tiny leftover pieces at the end
            chunks.append(chunk)
        start = end - overlap   # move forward but keep some overlap

    return chunks


chunks = chunk_text(raw_text, chunk_size=400, overlap=50)

print("number of chunks created:", len(chunks))
print()
print("sample chunk -->")
print(chunks[2])

number of chunks created: 13

sample chunk -->
ttributes describe the properties of an entity. Relationships describe how two or more entities are associated with each other. Cardinality defines the number of instances of one entity that can be associated with instances of another entity, common types being one-to-one, one-to-many and many-to-many.

Chapter 3: Normalization
Normalization is the process of organizing data in a database to reduc


## 4. Building the retriever (TF-IDF + cosine similarity)

For retrieval, decided against embedding models like sentence-transformers
since those need downloading fairly large models and take longer to set up.
TF-IDF is older and simpler but works fine for a small single topic document
like this one - each chunk gets turned into a vector of word importance
scores, then we compare the question's vector against every chunk vector
using cosine similarity.

Higher similarity score = chunk is more relevant to the question.

(If this was a bigger production style system I'd probably use proper
embeddings like all-MiniLM-L6-v2, but TF-IDF does the job here and is a lot
faster to get running.)

In [8]:
vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = vectorizer.fit_transform(chunks)

print("vocab size:", len(vectorizer.vocabulary_))
print("chunk vector matrix shape:", chunk_vectors.shape)

vocab size: 244
chunk vector matrix shape: (13, 244)


In [9]:
def retrieve_chunks(query, top_k=3):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, chunk_vectors)[0]

    # indexes of the top_k highest similarity scores
    top_indexes = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indexes:
        results.append({
            "chunk": chunks[idx],
            "score": round(float(similarities[idx]), 4)
        })
    return results


# quick test
test_results = retrieve_chunks("What is normalization?", top_k=3)
for r in test_results:
    print(r["score"], "-", r["chunk"][:90], "...")

0.3265 - ttributes describe the properties of an entity. Relationships describe how two or more ent ...
0.1412 - ncies are used to identify candidate keys and to guide the normalization process. A trivia ...
0.0 - ule is conflict serializable if it can be transformed into a serial schedule by swapping n ...


## 5. The Generator

This is the part where an actual language model writes the final answer,
using the retrieved chunks as context.

Went with `google/flan-t5-small` from Hugging Face for this - it's a small
instruction tuned model, works decently for answering questions from a given
context, and the big advantage is it's public so there's no API key or
account needed, it just downloads the weights the first time you run this
cell (needs internet for that one time download, after that it's cached
locally).

Also kept a plain fallback generator below in case the model can't load for
some reason (no internet at that moment, low RAM, etc) - it just extracts the
most relevant sentences from the retrieved chunks instead of the model
generating new text. Not true generation, but keeps the notebook usable even
without the model.

(if I had a better laptop/GPU I'd probably try flan-t5-base instead since it
gives noticeably better answers, small was mainly a speed choice for running
this on CPU)

In [10]:
from transformers import pipeline

try:
    generator = pipeline("text2text-generation", model="google/flan-t5-small")
    HAS_LLM = True
    print("flan-t5-small loaded successfully, will use it to generate answers")
except Exception as e:
    generator = None
    HAS_LLM = False
    print("could not load flan-t5 (probably no internet connection right now)")
    print("error was:", e)
    print("falling back to the extractive method instead")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

could not load flan-t5 (probably no internet connection right now)
error was: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"
falling back to the extractive method instead


In [11]:
def generate_with_llm(query, retrieved):
    context = " ".join([r["chunk"] for r in retrieved])
    context = context[:800]   # flan-t5 doesn't handle super long context well, trimming a bit

    prompt = f"Answer the question using only the given context.\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"

    output = generator(prompt, max_new_tokens=80)
    return output[0]["generated_text"].strip()

In [12]:
def generate_extractive(query, retrieved):
    # fallback answer generator, no model needed for this one
    # scores each sentence in the retrieved chunks by how many question
    # words it shares and picks the best matching ones

    full_context = " ".join([r["chunk"] for r in retrieved])
    sentences = re.split(r'(?<=[.!?])\s+', full_context)

    q_words = set(re.findall(r"\w+", query.lower()))
    stopwords = {"what","is","are","the","a","an","of","in","to","how","does","do","explain","define"}
    q_words = q_words - stopwords

    scored_sentences = []
    for sent in sentences:
        sent_words = set(re.findall(r"\w+", sent.lower()))
        overlap = len(q_words & sent_words)
        if overlap > 0:
            scored_sentences.append((overlap, sent.strip()))

    scored_sentences.sort(key=lambda x: x[0], reverse=True)

    if not scored_sentences:
        return "Sorry, I couldn't find a relevant answer for this in the document."

    best_sentences = [s for score, s in scored_sentences[:3]]
    return " ".join(best_sentences)

## 6. Putting it all together - the RAG pipeline

This function ties everything together - takes a question, calls the
retriever to get the relevant chunks, then sends those chunks to flan-t5 if
it loaded properly, otherwise uses the fallback.

In [13]:
def rag_answer(query, top_k=3, verbose=True):
    retrieved = retrieve_chunks(query, top_k=top_k)

    if verbose:
        print(f"Query: {query}")
        print("-" * 60)
        print("Top retrieved chunks:")
        for r in retrieved:
            print(f"  score={r['score']}  ->  {r['chunk'][:70]}...")
        print("-" * 60)

    if HAS_LLM:
        try:
            answer = generate_with_llm(query, retrieved)
        except Exception as e:
            print("model generation failed, falling back to extractive method. error:", e)
            answer = generate_extractive(query, retrieved)
    else:
        answer = generate_extractive(query, retrieved)

    return answer

## 7. Testing the system with a few questions

In [14]:
q1 = "What is normalization in DBMS?"
ans1 = rag_answer(q1)
print()
print("ANSWER:", ans1)

Query: What is normalization in DBMS?
------------------------------------------------------------
Top retrieved chunks:
  score=0.3481  ->  Database Management System Notes

Chapter 1: Introduction to DBMS
A da...
  score=0.2132  ->  ttributes describe the properties of an entity. Relationships describe...
  score=0.0922  ->  ncies are used to identify candidate keys and to guide the normalizati...
------------------------------------------------------------

ANSWER: Database Management System Notes

Chapter 1: Introduction to DBMS
A database management system (DBMS) is software that is used to create and manage databases. Unlike a traditional file system, a DBMS reduces data redundancy and improves data integrity through centraliz ttributes describe the properties of an entity. Chapter 3: Normalization
Normalization is the process of organizing data in a database to reduc ncies are used to identify candidate keys and to guide the normalization process.


In [15]:
q2 = "What are the ACID properties of a transaction?"
ans2 = rag_answer(q2)
print()
print("ANSWER:", ans2)

Query: What are the ACID properties of a transaction?
------------------------------------------------------------
Top retrieved chunks:
  score=0.2861  ->  a single logical unit of work. Transactions must satisfy the ACID prop...
  score=0.0771  ->  ttributes describe the properties of an entity. Relationships describe...
  score=0.0642  ->  ons do not interfere with each other. Durability ensures that once a t...
------------------------------------------------------------

ANSWER: Transactions must satisfy the ACID properties, which stand for atomicity, consistency, isolation and durability. Atomicity ensures that a transaction either completes fully or not at all. Consistency ensures the database remains in a valid state before and after the transaction.


In [16]:
q3 = "What is the difference between conflict serializability and normal serializability?"
ans3 = rag_answer(q3)
print()
print("ANSWER:", ans3)

Query: What is the difference between conflict serializability and normal serializability?
------------------------------------------------------------
Top retrieved chunks:
  score=0.2724  ->  ons do not interfere with each other. Durability ensures that once a t...
  score=0.2696  ->  ule is conflict serializable if it can be transformed into a serial sc...
  score=0.1633  ->  rtial dependency, meaning every non key attribute must depend on the w...
------------------------------------------------------------

ANSWER: Two operations conflict if they belong to different transactions, operate on the same data item and at least one of them is a write operation. Precedence graphs are used to test conflict serializability, where a cycle in the graph indicates the schedule is not serializable. The third normal form requires the table to be in second normal form and removes transitive dependency, meaning non key attributes should not depend on other non key attributes.


In [17]:
# a question whose answer isn't actually in the document, just checking
# that the system doesn't make something up when it has no relevant context
q4 = "What is the capital of France?"
ans4 = rag_answer(q4)
print()
print("ANSWER:", ans4)

Query: What is the capital of France?
------------------------------------------------------------
Top retrieved chunks:
  score=0.0  ->  otocols order transactions based on their timestamps to ensure seriali...
  score=0.0  ->  dicates the schedule is not serializable.

Chapter 8: Concurrency Cont...
  score=0.0  ->  ule is conflict serializable if it can be transformed into a serial sc...
------------------------------------------------------------

ANSWER: Sorry, I couldn't find a relevant answer for this in the document.


## 8. Quick evaluation table

Logging a few queries with their top retrieval score, mainly to check if the
retriever is actually pulling relevant chunks and not random ones. Not a
formal evaluation setup, just a sanity check.

In [18]:
eval_queries = [
    "What is an entity in ER model?",
    "Explain functional dependency",
    "What is two phase locking?",
    "What does BCNF mean?",
    "What happens during a deadlock?"
]

eval_rows = []
for q in eval_queries:
    top_chunk = retrieve_chunks(q, top_k=1)[0]
    eval_rows.append({
        "query": q,
        "top_score": top_chunk["score"],
        "matched_chunk_preview": top_chunk["chunk"][:60] + "..."
    })

eval_df = pd.DataFrame(eval_rows)
eval_df

,query,top_score,matched_chunk_preview
0,What is an entity in ER model?,0.5833,ancy and improves data integrity through centr...
1,Explain functional dependency,0.5431,"nctional dependency X to Y, X must be a super ..."
2,What is two phase locking?,0.4163,dicates the schedule is not serializable.\n\nC...
3,What does BCNF mean?,0.0000,otocols order transactions based on their time...
4,What happens during a deadlock?,0.6273,otocols order transactions based on their time...


## 9. Conclusion

Built a working RAG pipeline - chunking, TF-IDF based retrieval, and
generation using flan-t5-small (with a plain extractive fallback if the model
can't load). Tested on my own DBMS notes, retrieval picks up the right chunks
for most questions and the answers look reasonable, and it correctly says it
doesn't know when the question isn't covered in the document instead of
making something up.

**Things I'd improve if I had more time:**
- Chunk on sentence/paragraph boundaries instead of fixed character length -
  the fixed length splitting sometimes cuts a word in half right at the
  boundary (saw this in a couple of outputs above), splitting on blank lines
  or using a proper sentence tokenizer would fix that
- Use real sentence embeddings instead of TF-IDF for better semantic
  matching (right now it mostly matches on shared words, so it can struggle
  if a question is phrased very differently from how the notes are written)
- Try flan-t5-base instead of small for better quality answers, if running
  on a machine with more RAM/a GPU
- Add a re-ranking step after retrieval since the top TF-IDF match isn't
  always the actually most useful chunk
- Extend to handle multiple documents at once instead of just one file